In [59]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [60]:
filename='BASE_TARGET_20260622_alfin.csv'
df_base=cargar_archivo_csv(spark,filename,';',True)

# filename='20260612_BASE_CREDICASH_TARGET_TELEFONOS.csv'
# df_score=cargar_archivo_csv(spark,filename,';',True)


In [61]:
from pyspark.sql import functions as F

exprs = [
    F.count(F.when(F.col(c).isNotNull(), c)).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [62]:

df_base=df_base.withColumnRenamed('X_APPATERNO','APELLIDO_PATERNO')
df_base=df_base.withColumnRenamed('X_APMATERNO','APELLIDO_MATERNO')
df_base=df_base.withColumnRenamed('CampaÃ±a','campania')
df_base=df_base.withColumnRenamed('X_NOMBRE','NOMBRES')
df_base=df_base.withColumnRenamed('DNI','NUMERO_DOCUMENTO')
df_base=df_base.withColumnRenamed('SUCURSAL_COMERCIAL','sucursal_comercial')
df_base=df_base.withColumnRenamed('FLAG_DEUDA_V_OFERTA','flag_deuda_v_oferta')
df_base=df_base.withColumnRenamed('REGION_COMERCIAL','Region_comercial')
df_base=df_base.withColumnRenamed('TIPO_CLIENTE_COMERCIAL','tipo_cliente_riegos')
df_base = df_base.withColumn("Campana", F.lit("202606"))
df_base = df_base.withColumn("lote", F.lit("BASE 2026-06-23"))
df_base = df_base.withColumn("cl_base", F.lit("Junio 2026"))
df_base = df_base.withColumn("cl_carga", F.lit("2026-06-23"))
df_base = df_base.withColumn("cl_estado", F.lit("1"))
df_base = df_base.withColumn("estado", F.lit("ACTIVO"))


In [63]:
cols_prueba = set(df_base.columns)

cols_formato = set(
    """cl_id, TIPO_DOI, NUMERO_DOCUMENTO, NOMBRES, APELLIDO_PATERNO, APELLIDO_MATERNO, SUCURSAL, TIENDA, DEPARTAMENTO, PROVINCIA, DISTRITO, FEC_NACIMIENTO, OFERTA_MAX, OFERTA_REEN, Tipo_verificacion, GRUPO_RIESGO, proveedor, lote, estado, Tasa_1, Tasa_2, Tasa_3, Tasa_4, Tasa_5, Tasa_6, Tasa_7, segmento, Campana, PLAZO, TEM, PROPENSION_IC, Desgravamen, CUOTA, Edad, Oferta_12M, Tasa_12M, Desgravamen_12M, CUOTA_12M, Oferta_18M, Tasa_18M, Desgravamen_18M, CUOTA_18M, Oferta_24M, Tasa_24M, Desgravamen_24M, CUOTA_24M, Oferta_36M, Tasa_36M, Desgravamen_36M, CUOTA_36M, Validador_Telefono, Prioridad, Nombre_prioridad, Deuda_1, Entidad_1, Deuda_2, Entidad_2, Deuda_3, Entidad_3, sucursal_comercial, Agencia_comercial, Region_comercial, Ubicacion, OfertaMaximaSinSeguro, color, color_final, PROPENSION, OFERTA_FINAL, GARANTIA, Oferta_Minima_Paperless, RANGO_OFERTA, RANGO_SUELDO, CAPACIDAD_MAX, PEER, PROP_COMER, TIPO_GEST, CLIENTE_NUEVO, GRUPO_TASA, NUEVOS_3M, NUEVOS_6M, NUEVOS_9M, NUEVOS_12M, NUEVOS_4M, GRUPO_MONTO, TASA_VS_MONTO, USUARIO, incremento_monto_riesgos, FLG_DEUDA_PLUS, tipo_cliente_riegos, USER_V3, LEAD_CALIDAD, SEGMENTO_USER, RANGO_EDAD, RANGO_OFERTA2, PERIODO, RETIRO_GEST, MEJOR_TIPIFICACION, STATUS, FECHA_SOL, BASE, RESULTADO, NUM_ENRIQUECIDO, TIPO_CONTACTO, Q_VENTAS, LOCALIDAD, DESEMBOLSADO, MONTO_DESEMBOLSADO, SBI, CRUCE, PREST_PREVIO, ID_CLIENTE, RANGO_EDAD2, Fecha_Envio, TIPO_BD, COD_BD, NOMB_BD, MES_GESTION, TIPO_CLIENTE, GRUPO_TASA_REENGANCHE, SALDO_DIFERENCIAL_REENG, FLAG_REENG, RETIRO_DESEMBOLSO, FRESCURA, flag_deuda_v_oferta, MGNEG, PERFIL_RO, TIPO_BASE, cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7, cl_telf8, cl_telf9, cl_telf10, cl_movil, cl_celular, cl_telefono, cl_turno, cl_gestor, cl_asesor, cl_accion, cl_gestion, cl_estado, cl_fecha_gestion, cl_hora_gestion, cl_hits, cl_fecha_llamar, cl_prioridad, cl_orden, cl_predictivo, cl_tiempo, cl_base, cl_mes, cl_carga, id_carga, cl_area, fecha_alimentacion, cl_base_ant, cl_accion_ant, cl_fecha_ant, campania, PROMOCION, PROMOCION2, nombre_base, NumEntidades, p_banco, PERFIL_GLOBAL, FLG_AAHH, SCORE_TELEFONO, PILOTO_PLAZAS, INTENSIDAD_MAX"""
    .replace("\n", "")
    .split(",")
)

# limpiar espacios
cols_formato = set(col.strip() for col in cols_formato)

# comparación
solo_en_prueba = cols_prueba - cols_formato

print("Solo en df_base:", solo_en_prueba)
# print(cols_formato)

Solo en df_base: {'VARIACION_OFERTA_CAMPAÃ‘A_ANTERIOR', 'VARIACION_TASA_CAMPAÃ‘A_ANTERIOR', 'CAMP_ADP', 'CAMP_BONO', 'tasa_minima'}


In [64]:
df_base=df_base.drop('tasa_minima', 'CAMP_ADP', 'VARIACION_TASA_CAMPAÃ‘A_ANTERIOR', 'CAMP_BONO', 'VARIACION_OFERTA_CAMPAÃ‘A_ANTERIOR')

In [65]:
df_base = df_base.withColumn(
    "NUMERO_DOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("NUMERO_DOCUMENTO")),
        F.lit(8)
    )
)

In [67]:
df_base_subir=df_base.select('NUMERO_DOCUMENTO').join(df_formato,['NUMERO_DOCUMENTO'],'leftanti')

In [68]:
overwrite_table_SQL(spark,df_base_subir,f'ref_alfin_dni',server_zeus,user_zeus,pwd_zeus,'ODIN')
overwrite_table_SQL(spark,df_base_subir,f'ref_alfin_dni',server_sa,user_sa,pwd_sa,'CRONOX')


In [ ]:
query = """
    SELECT *
    FROM OPENQUERY([192.168.3.90], '
        SELECT *
        FROM crm_target.alfin_clientes
        WHERE cl_base = ''junio 2026''
    ')
    """
df_formato=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

df_formato.count()

35

## desembolso

In [69]:
query = f"""
    SELECT *
        FROM (
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,1 as ordern 
            from VALENTINA.dbo.alfcc_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
            UNION ALL
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono as cl_telf1,  1 as desembolso,  2 as orden
            from VALENTINA.dbo.alfin_ventas a
            inner join cronox.dbo.ref_alfin_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
        ) t
        """
df_desembolso=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)
df_desembolso.count()

0

In [42]:

window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("fecha_llamada").desc())
df_desembolso = df_desembolso.withColumn("ref_01", row_number().over(window_spec))
df_desembolso = df_desembolso.filter(col("ref_01") == 1).drop('ref_01')

df_enriquecido=df_desembolso.select('NUMERO_DOCUMENTO', 'cl_telf1')

In [43]:
df_base=df_base.join(df_enriquecido,['NUMERO_DOCUMENTO'],'inner')

In [44]:
df_base.show(2)

+----------------+-------------+--------+----------------+----------------+---------------+----------+-----+------+-------------+-------------------+--------+-----------------------+-------------+--------------+----------------+------------+---------+------------+-----------------+--------------------+-------------------+-------------+------------+-----+------+------+------+------+------+------+------+----+----------+------------+------------+--------+---------+--------------+-------------+-------+---------------+----------+----------+---------+------+---------+
|NUMERO_DOCUMENTO|PROPENSION_IC| USER_V3|APELLIDO_PATERNO|APELLIDO_MATERNO|        NOMBRES|OFERTA_MAX|PLAZO| CUOTA|CAPACIDAD_MAX|tipo_cliente_riegos|campania|SALDO_DIFERENCIAL_REENG| TIPO_CLIENTE|   color_final|       TIPO_BASE|DEPARTAMENTO|PROVINCIA|    DISTRITO|Agencia_comercial|    Region_comercial|flag_deuda_v_oferta|   GRUPO_TASA| GRUPO_MONTO|MGNEG|Tasa_1|Tasa_2|Tasa_3|Tasa_4|Tasa_5|Tasa_6|Tasa_7|Edad|RANGO_EDAD|RANGO_OFER

## cet

In [70]:
query = f"""
    SELECT *
    FROM (
        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 6 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesCencosudTc] a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 4 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDiners a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 8 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesEfectiva a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.DNI_CLIENTE AS NUMERO_DOCUMENTO,a.Telefono_Llamado AS TELEFONO,a.ESTADOS AS TIPO_GESTION, a.Descripcion AS GESTION,a.FECHA_ENVIO, 7 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionAgentesEfectivaN a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.DNI_CLIENTE = b.NUMERO_DOCUMENTO
        WHERE a.Telefono_Llamado IS NOT NULL

        UNION ALL

        SELECT a.CODDOC AS NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.FECHA_ENVIO, 5 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesCencoPP a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.CODDOC = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL

        UNION ALL

        SELECT a.NUMERO_DOCUMENTO, a.TELEFONO, a.TIPO_GESTION, a.GESTION, a.fecha_llamada AS FECHA_ENVIO, 3 AS peso
        FROM [MAEBA].[ADM_OBJ_TG].tGestionMesDinerstc a
        INNER JOIN odin.dbo.ref_alfin_dni b
            ON a.NUMERO_DOCUMENTO = b.NUMERO_DOCUMENTO
        WHERE a.TELEFONO IS NOT NULL
    ) t
    """
df_maeba=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)


query = f"""
    SELECT *
        FROM (
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION ,2 as peso 
            FROM VALENTINA.dbo.alfin_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfin_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
            UNION ALL
            select 
            a.dni_cliente as NUMERO_DOCUMENTO,  
            a.celular as TELEFONO,
            a.Fecha as FECHA_ENVIO,
            b.nivel_1 AS TIPO_GESTION,
            b.nivel_2 AS GESTION, 1 as peso 
            FROM VALENTINA.dbo.alfcc_gestion a
            inner join cronox.dbo.ref_alfin_dni C
            on a.dni_cliente COLLATE Modern_Spanish_CI_AS=C.NUMERO_DOCUMENTO   
            LEFT JOIN VALENTINA.dbo.alfcc_tipificaciones B 
            ON A.Tipificacion = B.id_banco
            where a.fecha<'2026-06-01'
        ) t
        """
df_valentina=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

In [ ]:
df_cet=df_maeba.unionByName(df_valentina)


In [72]:

window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("peso").asc(),F.col("FECHA_ENVIO").desc())
df_cet = df_cet.withColumn("ref_01", row_number().over(window_spec))
df_cet = df_cet.filter(col("ref_01") == 1).drop('ref_01','peso')

In [74]:
print(df_cet.columns)

['NUMERO_DOCUMENTO', 'TELEFONO', 'TIPO_GESTION', 'GESTION', 'FECHA_ENVIO']


In [73]:
df_cet.count()

2355

In [ ]:
print([row['GESTION' ] for row in df_cet.select('GESTION').distinct().collect()])


In [75]:
df=df_cet.select('NUMERO_DOCUMENTO','TELEFONO')

In [77]:
df_base=df_base.join(df.select('NUMERO_DOCUMENTO',F.col('TELEFONO').alias('cl_telf1')),['NUMERO_DOCUMENTO'],'left')

In [ ]:
df_desembolso = df_desembolso.withColumn(
    "dif_meses",
    F.floor(
        F.months_between(
            F.to_date(F.lit('2026-05-01')),
            F.col("fecha_llamada")
        )
    )
)

df_desembolso=df_desembolso.withColumn('marca',when(F.col('dif_meses').isin(0,1),'INVENTARIO TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'INVENTARIO TARGET 1')
                                                .otherwise(F.lit('INVENTARIO TARGET 2'))
)

df_gestion=df_gestion.filter(F.col('nivel_2').isin(nivel_2))
df_gestion=df_gestion.filter(F.col('nivel_1').isin('CONTACTO EFECTIVO CON TITULAR'))
df_gestion = df_gestion.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)

df_gestion=df_gestion.withColumn('marca',when(F.col('dif_meses').isin(0,1),'CET TARGET 0')
                                                .when(F.col('dif_meses').isin(2,3,4),'CET TARGET 1')
                                                .otherwise(F.lit('CET TARGET 2'))
)

## desembolso

In [157]:
query = f"""
    SELECT *
        FROM (
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono,  1 as desembolso 
            from VALENTINA.dbo.alfcc_ventas a
            inner join cronox.dbo.ref_credicash_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
            UNION ALL
            select a.fecha as fecha_llamada,a.dni as NUMERO_DOCUMENTO,a.telefono,  1 as desembolso 
            from VALENTINA.dbo.alfin_ventas a
            inner join cronox.dbo.ref_credicash_dni b
                on a.dni COLLATE Modern_Spanish_CI_AS=b.NUMERO_DOCUMENTO
            where a.estado in(4,13,0)
            and a.fecha<'2026-06-01'
        ) t
        """
df_desembolso=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)



In [160]:
window_spec = Window.partitionBy("NUMERO_DOCUMENTO").orderBy(F.col("fecha_llamada").desc())
df_desembolso = df_desembolso.withColumn("ref_01", row_number().over(window_spec))
df_desembolso = df_desembolso.filter(col("ref_01") == 1).drop('ref_01')


In [159]:
df_enriquecido=df_desembolso.select('NUMERO_DOCUMENTO', 'telefono')

In [161]:
df_base_2=df_base.filter((F.col('cl_telf1')=='0')&(F.col('cl_telf2')=='0'))


In [163]:
df_base_2=df_base_2.join(df_enriquecido,['NUMERO_DOCUMENTO'],'left')
df_base_2=df_base_2.withColumn('cl_telf1',when(F.col('telefono')=='0',F.col('telefono'))).drop('telefono')

In [164]:
df=df_base_2.filter(F.col('cl_telf1')!='0').toPandas()

In [126]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

# query = """
# SELECT cl_telf1,cl_telf2,NUMERO_DOCUMENTO FROM alfcc_clientes
# where cl_telf1 is not null or cl_telf2 is not null
# """

# df_historico = pd.read_sql(query, engine_mysql)
# print(df_formato.columns.tolist())

In [165]:
df[df["PLAZO_ELECTRO"] == "nan"].shape

(0, 43)

In [132]:
df.shape

(8222, 42)

In [168]:

df.to_sql(
    name="alfcc_clientes",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

0

In [166]:
df=df.drop(columns='DEMANDA')
print(df.columns.tolist())

['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'RANGO_SUELDO', 'SITUACION_LABORAL', 'INTENSIDAD_MAX', 'CAMPANA', 'LOTE', 'cl_base', 'cl_carga', 'cl_estado', 'estado', 'FLG_PEARS', 'SCORE_TELEFONO', 'cl_telf1', 'cl_telf2']


In [167]:
import numpy as np
import pandas as pd

df["PLAZO_ELECTRO"] = (
    pd.to_numeric(df["PLAZO_ELECTRO"].replace("nan", np.nan), errors="coerce")
    .fillna(0)
    .astype(int)
)

df["PLAZO_CREDICASH"] = (
    pd.to_numeric(df["PLAZO_CREDICASH"].replace("nan", np.nan), errors="coerce")
    .fillna(0)
    .astype(int)
)

In [139]:
df["PLAZO_ELECTRO"] = (
    df["PLAZO_ELECTRO"]
    .replace("nan", np.nan)
    .fillna(0)
    .astype(int)
)
df["PLAZO_CREDICASH"] = (
    df["PLAZO_CREDICASH"]
    .replace("nan", np.nan)
    .fillna(0)
    .astype(int)
)

In [ ]:
['NUMERO_DOCUMENTO', 'TIPO_DOCUMENTO', 'NOMBRE', 'PERFIL', 'OFERTA_CREDICASH', 'PLAZO_CREDICASH', 'TASA_CREDICASH', 'CME_CREDICASH', 'GRUPO_SEGMENTO', 'PROPENSION_CREDICASH', 'OFERTA_ELECTRO', 'PLAZO_ELECTRO', 'TASA_ELECTRO', 'CME_ELECTRO', 'PROPENSION_ELECTRO', 'Region', 'Tienda_IR', 'Semaforo', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'DIRECCION', 'FLG_AAHH', 'FLG_URB', 'TIPO_ZONA', 'MARCA_PD', 'PRODUCTO_EXTERNO', 'FLG_CRUCE_RECURRENTE', 'FRESCURA', 'RANGO_SUELDO', 'SITUACION_LABORAL', 'INTENSIDAD_MAX', 'CAMPANA', 'LOTE', 'cl_base', 'cl_carga', 'cl_estado', 'estado', 'FLG_PEARS', 'SCORE_TELEFONO', 'cl_telf1', 'cl_telf2']


In [ ]:


df_desembolso = df_desembolso.withColumn(
    "dif_meses",
    F.floor(
        F.months_between(
            F.to_date(F.lit('2026-06-01')),
            F.col("fecha_llamada")
        )
    )
)
df_desembolso = df_desembolso.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)
query = """
  SELECT distinct dni_cliente FROM cronox.dbo.borrar_alfin_01
    """
df_list = obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)


df_list = df_list.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )

)
df_desembolso=df_desembolso.join(df_list,['dni_cliente'],'inner')
df_desembolso.count()

In [ ]:
query = f"""
    SELECT DISTINCT dni_cliente,contacto,retiro4
        FROM (
            SELECT dni_cliente,celular as contacto,
            case
                when tipificacion in (4,5) then 'seguimiento -1'
                else 'no llamar -1'
            end as retiro4
            FROM valentina.dbo.alfin_gestion
            WHERE DNI_ejecutivo!='99999999' 
            and tipificacion in ('4','5','16','17')
            and CAST(fecha AS DATE) >= DATEADD(MONTH, -1, CAST('{fecha_mes_base}' AS DATE))
            AND CAST(fecha AS DATE) < DATEADD(MONTH, 0, CAST('{fecha_mes_base}' AS DATE))
            UNION ALL
            SELECT DISTINCT dni_cliente,CELULAR as contacto, 'excluir' as retiro4
            FROM valentina.dbo.Excluir_Gestion_TARGET where servicio in ('ALFIN','ALFCC')
        ) t
        """
df_retiro_dni_contacto=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)



In [ ]:

import pandas as pd

df_formato_base['cl_telf1'] = df_formato_base['cl_telf1'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_formato_base['cl_telf2'] = df_formato_base['cl_telf2'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_formato_base['cl_telf2'] = df_formato_base['cl_telf2'].fillna(0)
df_formato_base['cl_telf1'] = df_formato_base['cl_telf1'].fillna(0)
ruta_archivo = os.path.join(ruta_csv, 'fc_alfin.csv')

df_formato_base.to_csv(
    ruta_archivo,
    index=False,
    sep=';'
)

In [179]:
df_base=df_base.select('CAMPANA','FLG_PEARS','FRESCURA','OFERTA_CREDICASH','OFERTA_ELECTRO','PROPENSION_CREDICASH','PROPENSION_ELECTRO','Region','Semaforo','TASA_CREDICASH','NUMERO_DOCUMENTO','PRODUCTO_EXTERNO')

In [180]:
print([row['Campana' ] for row in df_base.select('Campana').distinct().collect()])
print([row['FLG_PEARS' ] for row in df_base.select('FLG_PEARS').distinct().collect()])
print([row['FRESCURA' ] for row in df_base.select('FRESCURA').distinct().collect()])
print([row['OFERTA_CREDICASH' ] for row in df_base.select('OFERTA_CREDICASH').distinct().collect()])
print([row['PRODUCTO_EXTERNO' ] for row in df_base.select('PRODUCTO_EXTERNO').distinct().collect()])
print([row['PROPENSION_CREDICASH' ] for row in df_base.select('PROPENSION_CREDICASH').distinct().collect()])
print([row['PROPENSION_ELECTRO' ] for row in df_base.select('PROPENSION_ELECTRO').distinct().collect()])
print([row['Region' ] for row in df_base.select('Region').distinct().collect()])
print([row['Semaforo' ] for row in df_base.select('Semaforo').distinct().collect()])
print([row['TASA_CREDICASH' ] for row in df_base.select('TASA_CREDICASH').distinct().collect()])


['202606']
[1, 0]
['3', '0', '1', '4', '2']
['7300.0', '11000.0', '3300.0', '4600.0', '8900.0', '4700.0', '12300.0', '1700.0', '2700.0', '8500.0', '2600.0', '4800.0', '14100.0', '13800.0', '12500.0', '2500.0', '12200.0', '13300.0', '2900.0', '4100.0', '13100.0', '1900.0', '5600.0', '10300.0', '11600.0', '14900.0', '8800.0', '9800.0', '9200.0', '8400.0', '9100.0', '5000.0', '3000.0', '5200.0', '10600.0', '14000.0', '7800.0', '7000.0', '4300.0', '2000.0', '13900.0', '10500.0', '8700.0', '6800.0', '12600.0', '4900.0', '4400.0', '9000.0', '14400.0', '6700.0', '10700.0', '10000.0', '8000.0', '12000.0', '3100.0', '8200.0', '5400.0', '12700.0', '3900.0', '9500.0', '9600.0', '7500.0', '11100.0', '14700.0', '4500.0', '8300.0', '6900.0', '11900.0', '3400.0', '9400.0', '10100.0', '11800.0', '14300.0', '6300.0', '3700.0', '14600.0', '4000.0', '7200.0', '11300.0', '12400.0', '7600.0', '6200.0', '2100.0', '10400.0', '14800.0', '2800.0', '13000.0', '6600.0', '7900.0', '13400.0', '5100.0', '10800.0', 

In [181]:

df_base=df_base.withColumn('mes',F.lit(6))

In [182]:
query = f"""
    select CAMPANA,FLG_PEARS,FRESCURA,OFERTA_CREDICASH,OFERTA_ELECTRO,PROPENSION_CREDICASH,PROPENSION_ELECTRO,Region,Semaforo,TASA_CREDICASH,NUMERO_DOCUMENTO,PRODUCTO_EXTERNO,5 as mes from VALENTINA.dbo.alfcc_clientes 
    where cl_base='mayo 2026'
    and lote='BASE 2026-05-01'
    """
df_base_anterior=obtener_tabla_sql(spark,query,server_sa,user_sa,pwd_sa,db_sa)

In [183]:
print([row['Campana' ] for row in df_base_anterior.select('Campana').distinct().collect()])
print([row['FLG_PEARS' ] for row in df_base_anterior.select('FLG_PEARS').distinct().collect()])
print([row['FRESCURA' ] for row in df_base_anterior.select('FRESCURA').distinct().collect()])
print([row['OFERTA_CREDICASH' ] for row in df_base_anterior.select('OFERTA_CREDICASH').distinct().collect()])
print([row['PRODUCTO_EXTERNO' ] for row in df_base_anterior.select('PRODUCTO_EXTERNO').distinct().collect()])
print([row['PROPENSION_CREDICASH' ] for row in df_base_anterior.select('PROPENSION_CREDICASH').distinct().collect()])
print([row['PROPENSION_ELECTRO' ] for row in df_base_anterior.select('PROPENSION_ELECTRO').distinct().collect()])
print([row['Region' ] for row in df_base_anterior.select('Region').distinct().collect()])
print([row['Semaforo' ] for row in df_base_anterior.select('Semaforo').distinct().collect()])
print([row['TASA_CREDICASH' ] for row in df_base_anterior.select('TASA_CREDICASH').distinct().collect()])

['202605']
['0', '1']
[1, 3, 4, 2, 0]
[5300, 9900, 4900, 11500, 3000, 11800, 4000, 6500, 13500, 5100, 1700, 2200, 7800, 8500, 6300, 4700, 9700, 4200, 10800, 3500, 7600, 6100, 12000, 10600, 3200, 5500, 7400, 13000, 8200, 14600, 5900, 10900, 2600, 5000, 7500, 12500, 14500, 3700, 8800, 13300, 6900, 11900, 10000, 8700, 6600, 14800, 6700, 9300, 13200, 11700, 13800, 1800, 10100, 11400, 14300, 13700, 10300, 13900, 12400, 1600, 5800, 10200, 4500, 3300, 3900, 6400, 5600, 8300, 12700, 12100, 5200, 12300, 3800, 14900, 14700, 5400, 2800, 8400, 7200, 10400, 6200, 7100, 3600, 11200, 9600, 4100, 11300, 15000, 10700, 7900, 13600, 9800, 4400, 7700, 6000, 9200, 2300, 2100, 12800, 11000, 8100, 14400, 8600, 4300, 9000, 7300, 12900, 2400, 3400, 14200, 4800, 3100, 14000, 2000, 5700, 12200, 9100, 6800, 11600, 9400, 13400, 8000, 10500, 9500, 2900, 11100, 2500, 12600, 8900, 4600, 1900, 2700, 7000, 14100, 13100]
['CREDICASH R / ELECTRO MSI', 'CREDICASH R / ELECTRO', 'CREDICASH R', 'CREDICASH / ELECTRO', 'CREDIC

In [134]:
df_tot=df_base.unionByName(df_base_anterior)

Aca empieza

In [ ]:
from pyspark.sql import functions as F

cols_finales = [
    "CAMPANA",
    "FLG_PEARS",
    "FRESCURA",
    "OFERTA_CREDICASH",
    "OFERTA_ELECTRO",
    "PROPENSION_CREDICASH",
    "PROPENSION_ELECTRO",
    "Region",
    "Semaforo",
    "TASA_CREDICASH",
    "NUMERO_DOCUMENTO",
    "PRODUCTO_EXTERNO",
    "mes"
]

def limpiar_base(df):
    df = df.select(*cols_finales)

    cols_string = [
        "CAMPANA",
        "FLG_PEARS",
        "FRESCURA",
        "OFERTA_CREDICASH",
        "OFERTA_ELECTRO",
        "PROPENSION_CREDICASH",
        "PROPENSION_ELECTRO",
        "Region",
        "Semaforo",
        "TASA_CREDICASH",
        "NUMERO_DOCUMENTO",
        "PRODUCTO_EXTERNO"
    ]

    for c in cols_string:
        df = df.withColumn(
            c,
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c).cast("string")) == "") |
                (F.lower(F.trim(F.col(c).cast("string"))) == "nan") |
                (F.lower(F.trim(F.col(c).cast("string"))) == "none") |
                (F.lower(F.trim(F.col(c).cast("string"))) == "null"),
                F.lit("no_aplica")
            ).otherwise(F.trim(F.col(c).cast("string")))
        )

    cols_quitar_decimal = [
        "FRESCURA",
        "OFERTA_CREDICASH",
        "OFERTA_ELECTRO",
        "PROPENSION_CREDICASH",
        "PROPENSION_ELECTRO",
        "TASA_CREDICASH"
    ]

    for c in cols_quitar_decimal:
        df = df.withColumn(
            c,
            F.when(F.col(c) == "no_aplica", F.col(c))
             .otherwise(F.regexp_replace(F.col(c), r"\.0$", ""))
        )

    df = df.withColumn("mes", F.col("mes").cast("int"))

    return df

df_mayo_limpio = limpiar_base(df_base)


In [22]:
df.groupBy('PRODUCTO_EXTERNO') \
    .count() \
    .orderBy('PRODUCTO_EXTERNO') \
    .show(30,truncate=False)


+-------------------------+-----+
|PRODUCTO_EXTERNO         |count|
+-------------------------+-----+
|CREDICASH R              |2210 |
|CREDICASH R / ELECTRO    |9799 |
|CREDICASH R / ELECTRO MSI|1822 |
+-------------------------+-----+



In [24]:
df_ref=df.toPandas()

In [26]:
df_ref.shape

(13831, 45)

In [19]:
print(198327+13831)
print(13831/212158)

212158
0.06519197956240161


In [ ]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT cl_telf1,cl_telf2,NUMERO_DOCUMENTO FROM alfcc_clientes
where cl_telf1 is not null or cl_telf2 is not null
"""

df_historico = pd.read_sql(query, engine_mysql)
print(df_formato.columns.tolist())

['cl_id', 'TIPO_DOI', 'NUMERO_DOCUMENTO', 'NOMBRES', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'SUCURSAL', 'TIENDA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'FEC_NACIMIENTO', 'OFERTA_MAX', 'OFERTA_REEN', 'Tipo_verificacion', 'GRUPO_RIESGO', 'PILOTO_PLAZAS', 'proveedor', 'lote', 'estado', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'segmento', 'Campana', 'PLAZO', 'TEM', 'PROPENSION_IC', 'Desgravamen', 'CUOTA', 'Edad', 'Oferta_12M', 'Tasa_12M', 'Desgravamen_12M', 'CUOTA_12M', 'Oferta_18M', 'Tasa_18M', 'Desgravamen_18M', 'CUOTA_18M', 'Oferta_24M', 'Tasa_24M', 'Desgravamen_24M', 'CUOTA_24M', 'Oferta_36M', 'Tasa_36M', 'Desgravamen_36M', 'CUOTA_36M', 'Validador_Telefono', 'Prioridad', 'Nombre_prioridad', 'Deuda_1', 'Entidad_1', 'Deuda_2', 'Entidad_2', 'Deuda_3', 'Entidad_3', 'sucursal_comercial', 'Agencia_comercial', 'Region_comercial', 'Ubicacion', 'OfertaMaximaSinSeguro', 'color', 'color_final', 'PROPENSION', 'OFERTA_FINAL', 'GARANTIA', 'Oferta_Minima_Paperless', '

In [175]:
filename='fc_alfin.csv'

filePath = os.path.join(ruta_csv, filename)

df_carga = pd.read_csv(filePath,sep=';')

df_carga["NUMERO_DOCUMENTO"] = (
    df_carga["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
print(df_carga.columns.tolist())


['DEPARTAMENTO', 'FLG_AAHH', 'NOMBRES', 'Agencia_comercial', 'CAPACIDAD_MAX', 'SALDO_DIFERENCIAL_REENG', 'GRUPO_MONTO', 'RANGO_OFERTA', 'cl_carga', 'NUMERO_DOCUMENTO', 'PLAZO', 'APELLIDO_PATERNO', 'SCORE_TELEFONO', 'RANGO_EDAD', 'oferta_max', 'cl_base', 'TIPO_CLIENTE', 'cl_estado', 'TIENDA', 'color_final', 'GRUPO_TASA', 'lote', 'Tasa_6', 'Tasa_5', 'cl_telf2', 'cl_telf1', 'Tasa_3', 'Tasa_7', 'flag_deuda_v_oferta', 'campania', 'USER_V3', 'TIPO_BASE', 'PROVINCIA', 'RANGO_SUELDO', 'DISTRITO', 'Region_comercial', 'FRESCURA', 'tipo_cliente_riegos', 'PROPENSION', 'nombre_base', 'Tasa_2', 'Tasa_4', 'PILOTO_PLAZAS', 'INTENSIDAD_MAX', 'MGNEG', 'Campana', 'CUOTA', 'estado', 'Edad', 'PROPENSION_IC', 'APELLIDO_MATERNO', 'Tasa_1']


In [89]:

df_formato_base["NUMERO_DOCUMENTO"] = (
    df_formato_base["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)


In [90]:

df_formato_base_inventario["NUMERO_DOCUMENTO"] = (
    df_formato_base_inventario["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)


In [126]:
df_formato_base['FLG_PEARS'] = (
    df_formato_base['FLG_PEARS']
    .astype(str)
    .str.upper()
    .map({'FALSE': 0, 'TRUE': 1})
    .fillna(0)
    .astype(int)
)

df_formato_base_inventario['FLG_PEARS'] = (
    df_formato_base_inventario['FLG_PEARS']
    .astype(str)
    .str.upper()
    .map({'FALSE': 0, 'TRUE': 1})
    .fillna(0)
    .astype(int)
)

In [ ]:
df_formato_base['FLG_PEARS'] = df_formato_base['FLG_PEARS'].astype(int)
df_formato_base_inventario['FLG_PEARS'] = df_formato_base_inventario['FLG_PEARS'].astype(int)

In [128]:
df_formato_base = df_formato_base.drop(columns=['DEMANDA'])
df_formato_base_inventario = df_formato_base_inventario.drop(columns=['DEMANDA'])

In [29]:
from sqlalchemy import create_engine

engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [63]:
# df_ref = df_ref.drop(columns=['DEMANDA'])
# df_ref = df_ref.drop(columns=['DETALLE_OFERTA'])
# df_ref = df_ref.drop(columns=['DETALLE_TASA'])
df_ref = df_ref.drop(columns=['FLG_PEARS'])


In [35]:
df_ref['cl_telf2'] = (
    pd.to_numeric(df_ref['cl_telf2'], errors='coerce')
      .astype('Int64')
      .astype(str)
)

In [59]:
cols = ['cl_telf1', 'cl_telf2']

for c in cols:
    df_ref[c] = (
        pd.to_numeric(df_ref[c], errors='coerce')
        .astype('Int64')
        .astype(str)
        .replace('<NA>', '0')   # o None si prefieres
    )

In [57]:
df_ref['OFERTA_ELECTRO'] = df_ref['OFERTA_ELECTRO'].fillna(0)


In [54]:
print(df_ref["OFERTA_ELECTRO"].head())
print(df_ref["OFERTA_ELECTRO"].unique()[:20])

0       nan
1    7000.0
2       nan
3    7000.0
4       nan
Name: OFERTA_ELECTRO, dtype: object
['nan' '7000.0' '6000.0' '4320.0' '3510.0' '4770.0' '5580.0' '2430.0'
 '1980.0' '5310.0' '2790.0' '6300.0' '5000.0' '3690.0' '5220.0' '3150.0'
 '5940.0' '5760.0' '5040.0' '4500.0']


In [55]:
cols = [
    "OFERTA_ELECTRO",
    "PLAZO_ELECTRO",
    "TASA_ELECTRO",
    "CME_ELECTRO",
    "PROPENSION_ELECTRO"
]

for c in cols:
    df_ref[c] = pd.to_numeric(df_ref[c], errors="coerce")

In [61]:
df_ref[['cl_telf1','OFERTA_ELECTRO']].head()

,cl_telf1,OFERTA_ELECTRO
0,984771393,0
1,973784976,7000
2,961644603,0
3,940331848,7000
4,988285159,0


In [60]:
cols = ['OFERTA_ELECTRO', 'OFERTA_ELECTRO']

for c in cols:
    df_ref[c] = (
        pd.to_numeric(df_ref[c], errors='coerce')
        .astype('Int64')
        .astype(str)
        .replace('<NA>', '0')   # o None si prefieres
    )

In [39]:

import pandas as pd

df_ref['cl_telf1'] = df_ref['cl_telf1'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_ref['cl_telf2'] = df_ref['cl_telf2'].apply(
    lambda x: str(int(x)) if pd.notnull(x) else None
)

df_ref['cl_telf2'] = df_ref['cl_telf2'].fillna(0)
df_ref['cl_telf1'] = df_ref['cl_telf1'].fillna(0)

In [45]:
print(df_base.columns)

['NUMERO_DOCUMENTO', 'PROPENSION_IC', 'USER_V3', 'APELLIDO_PATERNO', 'APELLIDO_MATERNO', 'NOMBRES', 'OFERTA_MAX', 'PLAZO', 'CUOTA', 'CAPACIDAD_MAX', 'tipo_cliente_riegos', 'campania', 'SALDO_DIFERENCIAL_REENG', 'TIPO_CLIENTE', 'color_final', 'TIPO_BASE', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'Agencia_comercial', 'Region_comercial', 'flag_deuda_v_oferta', 'GRUPO_TASA', 'GRUPO_MONTO', 'MGNEG', 'Tasa_1', 'Tasa_2', 'Tasa_3', 'Tasa_4', 'Tasa_5', 'Tasa_6', 'Tasa_7', 'Edad', 'RANGO_EDAD', 'RANGO_OFERTA', 'RANGO_SUELDO', 'FRESCURA', 'FLG_AAHH', 'INTENSIDAD_MAX', 'PILOTO_PLAZAS', 'Campana', 'lote', 'cl_base', 'cl_carga', 'cl_estado', 'estado', 'cl_telf1']


In [78]:
df_ref=df_base.toPandas()

In [80]:
df_ref = df_ref[df_ref['cl_telf1'].notna()]

In [81]:
df_ref.shape

(2355, 47)

In [82]:

df_ref.to_sql(
    name="alfin_clientes",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

2355